# Projeto Semantix — EDA de desempenho educacional nas capitais

**Objetivo:** analisar o IDEB dos anos iniciais do Ensino Fundamental nas capitais brasileiras e no Distrito Federal, comparando 2021 e 2023 e usando o IOEB 2023 como indicador contextual.

**Atenção:** o IOEB possui componentes relacionados ao IDEB. A correlação apresentada é descritiva e não causal.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../data/base_capitais_educacao_2023.csv")
df.head()


## 1. Qualidade e estrutura dos dados

In [ ]:
print(df.shape)
display(df.isna().sum().to_frame("nulos"))
display(df.describe(numeric_only=True).round(2))


## 2. Variáveis derivadas

In [ ]:
df["delta_ideb"] = df["ideb_2023"] - df["ideb_2021"]
df["delta_ioeb"] = df["ioeb_2023"] - df["ioeb_2021"]

df[["capital","uf","ideb_2021","ideb_2023","delta_ideb","ioeb_2023","delta_ioeb"]].sort_values("delta_ideb", ascending=False)


## 3. Distribuição do IDEB 2023

In [ ]:
df["ideb_2023"].plot(kind="hist", bins=8, figsize=(10,6))
plt.title("Distribuição do IDEB 2023")
plt.xlabel("IDEB 2023")
plt.ylabel("Número de capitais/DF")
plt.show()


## 4. Variação 2021–2023

In [ ]:
d = df.dropna(subset=["delta_ideb"]).sort_values("delta_ideb")
d.plot.barh(x="capital", y="delta_ideb", figsize=(10,9), legend=False)
plt.axvline(0, linewidth=1)
plt.title("Variação do IDEB entre 2021 e 2023")
plt.xlabel("Δ IDEB")
plt.show()


## 5. IOEB × IDEB

In [ ]:
print("Correlação de Pearson:", round(df[["ioeb_2023","ideb_2023"]].corr().iloc[0,1], 3))

ax = df.plot.scatter(x="ioeb_2023", y="ideb_2023", figsize=(10,6))
for _, r in df.iterrows():
    ax.annotate(r["capital"], (r["ioeb_2023"], r["ideb_2023"]), fontsize=7, xytext=(3,3), textcoords="offset points")
plt.title("IOEB 2023 × IDEB 2023")
plt.show()


## 6. Segmentação exploratória com K-Means

In [ ]:
base = df.dropna(subset=["ideb_2021","ideb_2023"]).copy()
features = base[["ideb_2023","delta_ideb"]]
X = StandardScaler().fit_transform(features)

scores = {}
for k in [2,3,4]:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

scores


In [ ]:
best_k = max(scores, key=scores.get)
model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
base["cluster"] = model.fit_predict(X)

print("K escolhido:", best_k)
print("Silhouette:", round(scores[best_k], 3))
display(base.groupby("cluster")[["ideb_2023","delta_ideb"]].mean().round(2))


In [ ]:
ax = base.plot.scatter(x="ideb_2023", y="delta_ideb", figsize=(10,6))
for _, r in base.iterrows():
    ax.annotate(r["capital"], (r["ideb_2023"], r["delta_ideb"]), fontsize=7, xytext=(3,3), textcoords="offset points")
plt.axhline(0, linewidth=1)
plt.title("Segmentação exploratória")
plt.show()


## 7. Interpretação

A análise deve ser lida como diagnóstico exploratório. A segmentação não é um modelo de previsão e a associação entre IOEB e IDEB não é causal, pois os indicadores compartilham componentes.

Perguntas para a apresentação:

1. Onde o IDEB aumentou mais entre 2021 e 2023?
2. Onde houve queda?
3. Como a distribuição do IDEB se apresenta entre as capitais?
4. Quais padrões aparecem na segmentação?
5. Quais informações adicionais seriam necessárias para uma análise causal ou preditiva?
